# OMR demo inference server (Colab)

Loads fine-tuned LoRA adapter `NathanZ721/omr-monophonic` on top of `Qwen/Qwen2-VL-2B-Instruct` and exposes a public Flask `/infer` endpoint via ngrok for the local demo app.

**Steps**
1. Select a GPU runtime (T4 is enough).
2. Paste your ngrok authtoken in cell 3 (https://dashboard.ngrok.com/get-started/your-authtoken).
3. Run all cells.
4. Copy the printed public URL into the local demo UI / `COLAB_INFER_URL`.

In [ ]:
# @title 1) Install dependencies (uninstall Colab torchao that breaks PEFT)
import importlib.machinery
import importlib.metadata
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.49.0,<5.0.0",
    "accelerate",
    "peft",
    "qwen-vl-utils",
    "safetensors",
    "flask",
    "pyngrok",
    "pillow",
])
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"])

try:
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version = None

if torchao_version is not None or importlib.machinery.PathFinder.find_spec("torchao") is not None:
    raise RuntimeError(
        "torchao is still present. Use Runtime -> Restart session, then rerun from this cell."
    )

print("Dependencies ready.")

In [ ]:
# @title 2) Load Qwen2-VL-2B + OMR LoRA adapter
import re

import torch
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
ADAPTER_ID = "NathanZ721/omr-monophonic"
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 768 * 768

SYSTEM_PROMPT = "You are an optical music recognition system. Return only PrIMuS semantic tokens."
USER_PROMPT = (
    "Transcribe this monophonic sheet-music image into PrIMuS semantic format. "
    "Return only one space-separated token string with no explanation or markdown."
)

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_ID, is_trainable=False)
model.to(device).eval()

print(f"Loaded {BASE_MODEL_ID} + {ADAPTER_ID} on {device} ({dtype})")

In [ ]:
# @title 3) Inference helpers
from io import BytesIO


def make_messages():
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]


def clean_prediction(text: str) -> str:
    text = re.sub(r"```(?:\w+)?", "", text or "").replace("```", "")
    return " ".join(text.split())


def predict_pil(image: Image.Image) -> str:
    image = image.convert("RGB")
    image.load()
    prompt_text = processor.apply_chat_template(
        make_messages(),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[prompt_text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    generated = output_ids[:, prompt_length:]
    return clean_prediction(
        processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    )


def predict_bytes(image_bytes: bytes) -> str:
    with Image.open(BytesIO(image_bytes)) as source:
        return predict_pil(source)


print("Inference helpers ready.")

In [ ]:
# @title 4) Configure ngrok + start Flask /infer server
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
SERVER_PORT = 8000  # @param {type:"integer"}

import threading

from flask import Flask, jsonify, request
from pyngrok import ngrok

if not NGROK_AUTHTOKEN.strip():
    raise ValueError(
        "Set NGROK_AUTHTOKEN from https://dashboard.ngrok.com/get-started/your-authtoken"
    )

ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

app = Flask(__name__)


@app.after_request
def add_cors_headers(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization, ngrok-skip-browser-warning"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    return response


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"ok": True, "model": BASE_MODEL_ID, "adapter": ADAPTER_ID})


@app.route("/infer", methods=["POST", "OPTIONS"])
def infer():
    if request.method == "OPTIONS":
        return ("", 204)
    if "image" not in request.files:
        return jsonify({"error": "Expected multipart field 'image'"}), 400
    upload = request.files["image"]
    image_bytes = upload.read()
    if not image_bytes:
        return jsonify({"error": "Empty image upload"}), 400
    try:
        semantic = predict_bytes(image_bytes)
    except Exception as exc:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return jsonify({"error": type(exc).__name__, "detail": str(exc)}), 500
    return jsonify({"semantic": semantic})


def _run_flask():
    app.run(host="0.0.0.0", port=SERVER_PORT, threaded=True, use_reloader=False)


thread = threading.Thread(target=_run_flask, daemon=True)
thread.start()

tunnel = ngrok.connect(SERVER_PORT, "http")
public_url = tunnel.public_url.rstrip("/")
print("=" * 72)
print("Colab OMR server is live")
print(f"Public URL:  {public_url}")
print(f"Health:      {public_url}/health")
print(f"Infer:       POST {public_url}/infer  (multipart field: image)")
print("Paste the Public URL into the local demo app as COLAB_INFER_URL.")
print("=" * 72)